# 第三功能：64块统一d/s分析
只读已保存原件，不访问FPGA、不发送START。每次分析都使用新的输出目录。

In [ ]:
from pathlib import Path
import sys, json, re, uuid
sys.dont_write_bytecode=True
BASE=Path('/home/xilinx/jupyter_notebooks/test_10_14')
KIT=BASE/'B_RX_Workbench_v0_2_2026-10-24'
for name in ('workbench','image64','window_audit','b_receiver'):
    if name in sys.modules and Path(sys.modules[name].__file__).resolve().parent!=KIT.resolve():
        raise RuntimeError('存在其他版本分析模块，请换新内核')
sys.path.insert(0,str(KIT))
from workbench import verify_package,run
print('固定文件检查:',verify_package())
name=input('会话名 [session_00000027]：').strip() or 'session_00000027'
if re.fullmatch(r'session_[0-9a-f]{8}',name) is None:raise ValueError('会话名格式不符')
SESSION=BASE/'board_runs'/name
choice=input('对齐策略：1参考前缀 / 2仅CAP [1]：').strip() or '1'
if choice not in ('1','2'):raise ValueError('策略选项不符')
POLICY='reference_prefix' if choice=='1' else 'capture_only'
MAX_BITS=int(input('全64局部位错上界门槛 [0]：').strip() or '0')
MAX_FRAMING=int(input('全64局部停止位错上界门槛 [0]：').strip() or '0')
if min(MAX_BITS,MAX_FRAMING)<0:raise ValueError('门槛不能为负')
print('输入:',SESSION,'策略:',POLICY,'不会配置Overlay或发送START')


In [ ]:
from IPython.display import display, HTML
import html
OUTPUT=BASE/'ds_analysis_runs'/(SESSION.name+'_'+uuid.uuid4().hex[:12])
progress_view=display(HTML('准备读取已保存原件……'),display_id=True)
def show_progress(done,total,item):
    progress_view.update(HTML('<progress max="%d" value="%d"></progress> %d/%d · %s'%(total,done,done,total,html.escape(item['state']))))
RESULT=run(SESSION,OUTPUT,policy=POLICY,max_bits=MAX_BITS,max_framing=MAX_FRAMING,progress=show_progress)
print('状态:',RESULT['job_state'])
print('统一软件候选:',RESULT['global_minimax_candidate_set'])
print('达到门槛的区域:',RESULT['threshold_candidate_region'])
print('完整结果:',OUTPUT)


In [ ]:
from report_export import verify_output
from urllib.parse import quote
print(verify_output(OUTPUT))
relative=(OUTPUT/'report.html').relative_to(Path('/home/xilinx/jupyter_notebooks')).as_posix()
url='/files/'+quote(relative)
display(HTML('<a target="_blank" href="'+url+'">打开或下载独立交互报告</a>'))
print('若浏览器下载HTML，直接在电脑打开即可，无需启动网页服务。')
print('可以选择统一d/s、ITEM、相位和位，查看投票点并导出SVG/CSV/JSON。')


In [ ]:
from output_tools import pack
ARCHIVE=OUTPUT.with_suffix('.zip')
print(json.dumps(pack(OUTPUT,ARCHIVE),ensure_ascii=False,indent=2))
print('此ZIP是分析结果，不包含完整原始会话；请保留原件。')
